In [102]:
import requests
import json
import pandas as pd
import numpy as np
import os
from datetime import datetime

with open("keys.json") as f:
    data = json.loads(f.read())
    STEAM_DATA_API = data["STEAM_DATA_API"] 
    STEAM_WEB_API = data["STEAM_WEB_API"] 

BASE_URL = "https://www.steamwebapi.com/steam/"
DT_ISO8601_FORMAT = "%Y-%m-%d %H:%M:%S"

In [61]:
def call(method: str, call_path: str, extra_params: str | None = None, payload: dict | None = None, base_url = BASE_URL):
    """
    calls the Steam Web API, stores the response to /data/steam_web_api, returns (location of saved file, errorcode)
    """
    method = method.upper()
    body = ""
    headers = {}

    request_url = base_url + call_path + "?key=" + STEAM_WEB_API 

    output_path = f"data/steam_web_api/{call_path.replace('/', '_')}"
    output_path = output_path[:output_path.find("markets")].rstrip()

    if extra_params != None:
        request_url = request_url + "&" + extra_params
        output_path = output_path + "_" + extra_params + ".json"

    if payload:
        body = json.dumps(payload)


    if method != "GET" and payload:
        headers["Content-Type"] = "application/json"

    try:
        r = requests.request(
            method,
            request_url,
            headers=headers,
            data=body.encode() if body else None,
            timeout=15,
        )
        r.raise_for_status()
        data_json = r.json()

        if len(data_json) == 0:
            return output_path, "RESPONSE EMPTY"
        
        with open(output_path, "x") as f:
            json.dump(data_json, f, indent=4)

        return output_path, None
    except requests.RequestException as e:
        detail = e.response.text if e.response is not None else "no body"
        return output_path, f"{e} — {detail}"


def getJsonFileasFlatMappedDataFrame(file_path):
    return pd.json_normalize(json.loads(open(file_path).read()))

def getJsonObjAsFlatMappedDataFrame(object):
    return pd.json_normalize(object)

def getData(call_name: str, file_name: str):
    """
    calls the Steam Data API, stores the response to /data/steam_data_api, returns (location of saved file, errorcode)
    """
    headers = {
        "Authorization": f"Bearer {STEAM_DATA_API}",
    }

    try:
        response = requests.get(
            call_name,
            headers=headers,
        )

        response.raise_for_status()
        data = response.json()
        file_name = file_name + "_" + str(datetime.now()) + ".json"
        with open(file_name, "x") as f:
            json.dump(data,f, indent=4)

        print(f"File: {file_name}" +  ", Size: " + str(os.path.getsize(file_name)/1000) + "KB" )
        return 0
    
    except requests.RequestException as e:
        detail = e.response.text if e.response is not None else "no body"
        print(f"{e} — {detail}")
        return 1

# API Aufrufe

In [ ]:
call_path = "api/items"
game_param = "game=cs2&max=100"

for pagenum in range(0,1):
    full_params = game_param + '&' + 'page=' + str(pagenum)
    location, errorcode = call("GET", call_path, full_params)
    if (errorcode != None):
        print(errorcode)
        break
    print(f"Nr: {pagenum}, File: {location}" +  ", Size: " + str(os.path.getsize(location)/1000) + "KB")


Nr: 0, File: data/steam_web_api/api_item_game=cs2&max=100&page=0.json, Size: 534.119MB


In [148]:
call_path = "https://steamdataapi.com/api/v1/items/all?game=cs2&markets=1&variants=1&meta=1&currency=USD"
file_name = "data/steam_data_api/all_items_meta"

errorcode = getData(call_path, file_name)

File: data/steam_data_api/all_items_meta_2026-11-19 18:45:12.163570.json, Size: 148210.956KB


In [ ]:
for pageNum in range(1,62):
    call_path = f"https://steamdataapi.com/api/v1/buy-orders?game=cs2&limit=500&page={pageNum}&currency=USD"
    file_name = f"data/buy_orders_page_{pageNum}"
    errorcode = getData(call_path, file_name)
    if errorcode == 1:
        break

# Datenexploration der Verkaufsangebote

In [ ]:
def remove_0_prices(df: pd.DataFrame):
    """
    Aufgrund von abrundung werden manche preise > 1 cent als 0 zurückgegeben
    -> bei kauf würden diese aber auf 1 cent aufgerundet werden
    """
    df.loc[df["price"] == 0, 'price'] = 1

def organize_columns(df: pd.DataFrame):
    """
    Löschen von unnötigen Spalten & umbenennung
    """
    df = df[["marketHashName","sold24h","markets","variants"]]
    df = df.rename(columns={"sold24h":"steam_sales_last_24h"})
    return df

def add_captured_at_to_variants_df(df: pd.DataFrame):
    df = df.explode("markets").dropna(subset=["markets"]).reset_index(drop=True)
    market_details = pd.json_normalize(df["markets"])
    variant_cols = "capturedAt"
    return pd.concat([df.drop(columns=["markets"]), market_details[variant_cols]], axis=1).drop_duplicates(subset="marketHashName")


def divide_df_based_on_variants(df: pd.DataFrame):
    """
    Trennt items in zwei dataframes, eins enthält json object mit variants informationen, anderes enthält keine variants 
    """
    has_variants = df["variants"].notna() & (df["variants"].str.len() > 0)
    df_no_var = df[~has_variants].drop(columns=["variants"])
    df_no_var["variant"] = "N"

    df_var = df[has_variants]
    df_var = add_captured_at_to_variants_df(df_var)

    return df_no_var, df_var


def extract_variants(df: pd.DataFrame):
    """
    json objekt in variants spalte wird exploded & normalisiert -> eine zeile pro variante
    """
    df = df.explode("variants").reset_index(drop=True)
    variant_details = pd.json_normalize(df["variants"])
    variant_cols = [c for c in ["variant", "markets"] if c in variant_details.columns]
    return pd.concat([df.drop(columns=["variants"]), variant_details[variant_cols]], axis=1)

def extract_markets(df: pd.DataFrame):
    """
    markt dictionaries in markets spalte werden exploded & normalisiert -> eine zeile pro markt, da es unterschiedliche namensgebungen zwischen den varianten markets und den no_variant markets gibt, werden die namen normalisiert (updatedAt / capturedAt -> capturedAt)
    """
    df = df.explode("markets").dropna(subset=["markets"]).reset_index(drop=True)
    market_details = pd.json_normalize(df["markets"])
    return pd.concat(
        [
            df.drop(columns=["markets"]),
            market_details.drop(columns=["icon", "updatedAt"], errors="ignore"),
        ],
        axis=1,
    )

def normalize_sell_orders(df: pd.DataFrame):
    """
    1. schauen ob varianten existieren oder nicht
    2. wenn varianten existieren, exploden & normalisieren, jede variante hat wieder eigene markets liste
    3. varianten namen umbennen, da quantityUpdatedAt / updatedAt/capturedAt nicht gleich genannt werden 
    4. alle zusammenfügen & market exploden & normalisieren
    """
    #df = organize_columns(df)

    df_no_var, df_var = divide_df_based_on_variants(df)

    df_no_var = extract_markets(df_no_var)

    if not df_var.empty:
        df_var = extract_variants(df_var)
        df_var = extract_markets(df_var)

        combined_df = pd.concat([df_no_var, df_var], ignore_index=True)
    else:
        combined_df = df_no_var.reset_index(drop=True)

    remove_0_prices(combined_df)
    combined_df["capturedAt"] = pd.to_datetime(combined_df["capturedAt"], format='ISO8601', utc=True)  
    combined_df["priceUpdatedAt"] = pd.to_datetime(pd.Timestamp.now("UTC"), format='ISO8601', utc=True)  

    return combined_df


def normalize_sell_orders_drop_variants(df: pd.DataFrame):
    df = organize_columns(df)

    has_variants = df["variants"].notna() & (df["variants"].str.len() > 0)
    df = df[~has_variants].drop(columns=["variants"])
    df = extract_markets(df)
    remove_0_prices(df)
    df["capturedAt"] = pd.to_datetime(df["capturedAt"], format='ISO8601', utc=True)  
    df["priceUpdatedAt"] = pd.to_datetime(pd.Timestamp.now("UTC"), format='ISO8601', utc=True)  

    return df
       

In [150]:
items_meta_file = json.loads(open("data/steam_data_api/all_items_meta_2026-11-19 18:45:12.163570.json").read())
meta_orders = getJsonObjAsFlatMappedDataFrame(items_meta_file["data"])

In [152]:
meta_orders = normalize_sell_orders(meta_orders)

In [161]:
meta_orders[meta_orders["meta.stickerId"].notna()]

,marketHashName,game,sold24h,priceUpdatedAt,meta.rarity,meta.category,meta.weaponType,meta.collection,meta.wear,meta.isStattrak,...,prices.median,prices.buyorder,prices.real,prices.realMedian,prices.realMarket,variant,market,price,quantity,capturedAt
199380,Sticker | 00 Nation (Glitter) | Rio 2022,cs2,11.0,2026-11-19 18:47:14.137486+00:00,Remarkable,Remarkable Sticker,Sticker,None,None,False,...,58.0,49.0,35.0,38.0,avan,N,avan,35,9.0,2026-11-19 18:43:04.375000+00:00
199381,Sticker | 00 Nation (Glitter) | Rio 2022,cs2,11.0,2026-11-19 18:47:14.137486+00:00,Remarkable,Remarkable Sticker,Sticker,None,None,False,...,58.0,49.0,35.0,38.0,avan,N,buff163,39,890.0,2026-11-19 18:44:22.556000+00:00
199382,Sticker | 00 Nation (Glitter) | Rio 2022,cs2,11.0,2026-11-19 18:47:14.137486+00:00,Remarkable,Remarkable Sticker,Sticker,None,None,False,...,58.0,49.0,35.0,38.0,avan,N,c5game,35,195.0,2026-11-19 18:44:22.556000+00:00
199383,Sticker | 00 Nation (Glitter) | Rio 2022,cs2,11.0,2026-11-19 18:47:14.137486+00:00,Remarkable,Remarkable Sticker,Sticker,None,None,False,...,58.0,49.0,35.0,38.0,avan,N,csfloat,37,162.0,2026-11-19 18:40:20.719000+00:00
199384,Sticker | 00 Nation (Glitter) | Rio 2022,cs2,11.0,2026-11-19 18:47:14.137486+00:00,Remarkable,Remarkable Sticker,Sticker,None,None,False,...,58.0,49.0,35.0,38.0,avan,N,csmoney,39,45.0,2026-11-19 18:44:01.606000+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320744,Sticker | zweih | Cologne 2026,cs2,4.0,2026-11-19 18:47:14.137486+00:00,High Grade,High Grade Sticker,Sticker,None,None,False,...,4.0,NaN,4.0,5.0,csfloat,N,mannco,7,1.0,2026-11-19 18:40:49.557000+00:00
320745,Sticker | zweih | Cologne 2026,cs2,4.0,2026-11-19 18:47:14.137486+00:00,High Grade,High Grade Sticker,Sticker,None,None,False,...,4.0,NaN,4.0,5.0,csfloat,N,marketcsgo,5,6.0,2026-11-19 18:42:00.915000+00:00
320746,Sticker | zweih | Cologne 2026,cs2,4.0,2026-11-19 18:47:14.137486+00:00,High Grade,High Grade Sticker,Sticker,None,None,False,...,4.0,NaN,4.0,5.0,csfloat,N,waxpeer,5,15.0,2026-11-19 18:42:41.775000+00:00
320747,Sticker | zweih | Cologne 2026,cs2,4.0,2026-11-19 18:47:14.137486+00:00,High Grade,High Grade Sticker,Sticker,None,None,False,...,4.0,NaN,4.0,5.0,csfloat,N,whitemarket,6,18.0,2026-11-19 18:30:10.828000+00:00


In [128]:
all_items_file_alt = json.loads(open("data/steam_data_api/all_items_no_info_2026-11-17 13:44:47.776898.json").read())
verkaufs_angebote_alt = getJsonObjAsFlatMappedDataFrame(all_items_file_alt["data"])

all_items_file_neu = json.loads(open("data/steam_data_api/all_items_no_info_2026-11-17 18:05:45.603960.json").read())
verkaufs_angebote_neu = getJsonObjAsFlatMappedDataFrame(all_items_file_neu["data"])

verkaufs_angebote_alt = normalize_sell_orders(verkaufs_angebote_alt)
verkaufs_angebote_neu = normalize_sell_orders(verkaufs_angebote_neu)

In [129]:
dt_verkaufs_angebote_alt = "2026-11-17 13:44:47"
dt_verkaufs_angebote_neu = "2026-11-17 18:05:45"

dt_verkaufs_angebote_alt = pd.to_datetime(datetime.strptime(dt_verkaufs_angebote_alt, DT_ISO8601_FORMAT), format='ISO8601', utc=True) 
dt_verkaufs_angebote_neu = pd.to_datetime(datetime.strptime(dt_verkaufs_angebote_neu, DT_ISO8601_FORMAT), format='ISO8601', utc=True) 

In [130]:
verkaufs_angebote_alt.head()

,marketHashName,steam_sales_last_24h,variant,market,price,quantity,capturedAt,priceUpdatedAt
0,'Blueberries' Buckshot | NSWC SEAL,17.0,N,avan,936,216.0,2026-11-17 13:43:09.578000+00:00,2026-11-19 18:31:27.478778+00:00
1,'Blueberries' Buckshot | NSWC SEAL,17.0,N,buff163,897,187.0,2026-11-17 13:44:25.468000+00:00,2026-11-19 18:31:27.478778+00:00
2,'Blueberries' Buckshot | NSWC SEAL,17.0,N,c5game,926,263.0,2026-11-17 13:44:25.468000+00:00,2026-11-19 18:31:27.478778+00:00
3,'Blueberries' Buckshot | NSWC SEAL,17.0,N,csfloat,930,190.0,2026-11-17 13:40:20.942000+00:00,2026-11-19 18:31:27.478778+00:00
4,'Blueberries' Buckshot | NSWC SEAL,17.0,N,csmoney,909,95.0,2026-11-17 13:44:01.373000+00:00,2026-11-19 18:31:27.478778+00:00


# 1. Vollständigkeit 

-> Wie viele einzigartige Skin+varianten haben Verkaufsangebote auf mehr als einem Markt

In [131]:
unique_columns = ["marketHashName", "variant", "market"]

# 1. Alle möglichen Zeilen
print(f"Count verkaufs_angebote_alt: {verkaufs_angebote_alt.shape[0]}")
print(f"Count verkaufs_angebote_neu: {verkaufs_angebote_neu.shape[0]} \n")

#2. Einzigarte Entries (name + variante + markt)
print(f"Count Angebote ohne Duplikate alt: {verkaufs_angebote_alt[unique_columns].drop_duplicates().shape[0]}")
print(f"Count Angebote ohne Duplikate neu: {verkaufs_angebote_neu[unique_columns].drop_duplicates().shape[0]} \n")

#3. Insgesamt alle möglichen skin + variant möglichkeiten
unique_columns = ["marketHashName", "variant"]
print(f"Count Alle skin+variant möglichkeiten alt: {verkaufs_angebote_alt[unique_columns].drop_duplicates().shape[0]}")
print(f"Count Alle skin+variant möglichkeiten neu: {verkaufs_angebote_neu[unique_columns].drop_duplicates().shape[0]} \n")

#3. Skin + variant möglichkeiten, mit Angeboten auf mind 2 platformen
print(f"Count Alle skin+variant möglichkeiten mit >= 2 Plattformen alt: {(verkaufs_angebote_alt.groupby(unique_columns)['market'].nunique() >= 2).shape[0]}")
print(f"Count Alle skin+variant möglichkeiten mit >= 2 Plattformen neu: {(verkaufs_angebote_neu.groupby(unique_columns)['market'].nunique() >= 2).shape[0]}")

Count verkaufs_angebote_alt: 392998
Count verkaufs_angebote_neu: 392415 

Count Angebote ohne Duplikate alt: 392998
Count Angebote ohne Duplikate neu: 392415 

Count Alle skin+variant möglichkeiten alt: 31807
Count Alle skin+variant möglichkeiten neu: 31809 

Count Alle skin+variant möglichkeiten mit >= 2 Plattformen alt: 31807
Count Alle skin+variant möglichkeiten mit >= 2 Plattformen neu: 31809


# 2. Eindeutingkeit
Gibt es duplikate für einen Eintrag ? (HashName + Variante + Markt)

In [132]:
subset_cols  = ['marketHashName', 'variant', 'market']

print(f" Duplizierte Einträge alt: {verkaufs_angebote_alt.duplicated(subset=subset_cols).sum()}")
print(f" Duplizierte Einträge neu: {verkaufs_angebote_neu.duplicated(subset=subset_cols).sum()}")

 Duplizierte Einträge alt: 0
 Duplizierte Einträge neu: 0


# 3. Plausibilität 

Wo liegen die Preisunterschiede? 
-> CS Markt sehr Volatil, also vergleich mit einer anderen Aggregator API

In [133]:
def get_aggregated_price_info(df: pd.DataFrame)-> pd.DataFrame:
     # Group pro item & Aggregate Prices, also build interesting statistical values
     price_stats = (df
                    .groupby(["marketHashName", "variant"], dropna=False)['price']
                    .agg(market_count = 'count',
                         min_price = 'min',
                         median_price = 'median',
                         max_price = 'max',
                         std_price = 'std').reset_index())


     price_stats['abs_spread'] = price_stats['max_price'] - price_stats['min_price']
     price_stats['rel_spread_pct'] = (price_stats['abs_spread'] / price_stats['min_price']) * 100
     price_stats['extreme_spreads'] = (price_stats['rel_spread_pct'] > 100)
     
     return price_stats

In [166]:
prices_alt = get_aggregated_price_info(verkaufs_angebote_alt)
extreme_spreads = prices_alt[prices_alt['extreme_spreads']]
print(f"Items with spread between Platforms over over 100% {extreme_spreads.shape[0]}")
print(f"Items without Spread: {prices_alt[prices_alt['rel_spread_pct'] <= 0].shape[0]}")
print(f"Average Spread across all items: {prices_alt['rel_spread_pct'].sum() / prices_alt['rel_spread_pct'].shape[0]}")  
print(f"Average Spread across all items: {prices_alt['rel_spread_pct'].sum() / prices_alt['rel_spread_pct'].shape[0]}")  
prices_alt.describe()

Items with spread between Platforms over over 100% 15368
Items without Spread: 1056
Average Spread across all items: 133.70414102119446
Average Spread across all items: 133.70414102119446


,market_count,min_price,median_price,max_price,std_price,abs_spread,rel_spread_pct
count,31807.000000,3.180700e+04,3.180700e+04,3.180700e+04,3.076300e+04,3.180700e+04,31807.000000
mean,12.355708,1.057447e+04,1.234946e+04,1.641237e+04,2.133925e+03,5.837902e+03,133.704141
std,4.190819,8.925666e+04,1.232195e+05,1.421137e+05,3.201236e+04,6.497376e+04,113.380308
min,1.000000,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,10.000000,2.900000e+01,3.800000e+01,6.300000e+01,7.972329e+00,2.200000e+01,36.790875
50%,14.000000,2.750000e+02,3.510000e+02,5.700000e+02,6.235711e+01,1.830000e+02,97.014925
75%,16.000000,3.556000e+03,4.287000e+03,5.774500e+03,5.397751e+02,1.645000e+03,210.282403
max,16.000000,7.500188e+06,1.500038e+07,1.514947e+07,4.553559e+06,8.474303e+06,400.000000


In [167]:
prices_alt.sort_values(by=["min_price"], ascending=True)

,marketHashName,variant,market_count,min_price,median_price,max_price,std_price,abs_spread,rel_spread_pct,extreme_spreads
16098,Sticker | 3DMAX | Budapest 2025,N,15,1,2.0,3,8.837151e-01,2,200.000000,True
16101,Sticker | 3DMAX | Shanghai 2024,N,13,1,2.0,3,8.006408e-01,2,200.000000,True
16102,Sticker | 910 (Embroidered) | Budapest 2025,N,8,1,2.5,5,1.407886e+00,4,400.000000,True
16157,Sticker | AMKAL ESPORTS | Copenhagen 2024,N,14,1,2.0,4,1.016350e+00,3,300.000000,True
16178,Sticker | ANNIHILATION | Paris 2023,N,11,1,2.0,4,8.738629e-01,3,300.000000,True
...,...,...,...,...,...,...,...,...,...,...
21229,Sticker | Team Dignitas (Holo) | Katowice 2014,N,6,2453091,2699242.0,3578862,4.090371e+05,1125771,45.891938,False
29519,★ Sport Gloves | Hedge Maze (Factory New),N,5,2700068,2940865.0,3100039,1.478198e+05,399971,14.813368,False
23774,Sticker | iBUYPOWER (Holo) | Katowice 2014,N,4,6375159,6679690.5,8999900,1.228233e+06,2624741,41.171381,False
21477,Sticker | Titan (Holo) | Katowice 2014,N,5,6675167,15000375.0,15149470,4.553559e+06,8474303,126.952674,True


## Pretty big spreads, compare with data from other source (steam web api)

In [140]:
def extract_prices(df: pd.DataFrame):
    df = df.explode("prices").dropna(subset=["prices"]).reset_index(drop=True)
    price_details = pd.json_normalize(df["prices"], max_level=0)
    price_cols = [c for c in ["price", "source", "quantity", "type", "created_at", "variants"] if c in price_details.columns]
    return pd.concat([df.drop(columns=["prices"]), price_details[price_cols]], axis=1)

def extract_variants(df: pd.DataFrame):
    df = df.drop(columns=["price"])
    exploded = df.assign(variants=df["variants"].apply(lambda d: list(d.items()) if isinstance(d, dict) else [])).explode("variants")
    exploded[["variant", "price"]] = pd.DataFrame(exploded["variants"].tolist(), index=exploded.index)
    return exploded.drop(columns=["variants"])

def normalize_web_api_items(df: pd.DataFrame):
    df = df[["markethashname", "priceupdatedat.date", "prices"]]
    df = df.rename(columns={"priceupdatedat.date": "priceupdatedat"})
    has_info = (df["prices"].notna() & (df["prices"].str.len() > 0) & df["priceupdatedat"].notna())
    df = df[has_info]
    
    df = extract_prices(df)

    has_variants = df["variants"].notna()
    no_var = df[~has_variants]
    var = df[has_variants]

    no_var = no_var.drop(columns=["variants"])
    no_var["variant"] = "N"
    
    if not var.empty:
        var = extract_variants(var)
        combined_df = pd.concat([no_var, var], ignore_index=True)
    else:
        combined_df = no_var.reset_index(drop=True)

    remove_0_prices(combined_df)
    combined_df["created_at"] = pd.to_datetime(combined_df["created_at"], format='ISO8601', utc=True)    
    combined_df["priceupdatedat"] = pd.to_datetime(combined_df["priceupdatedat"], format='ISO8601', utc=True)  
    
    combined_df = combined_df.rename(columns={"priceupdatedat": "capturedAt", "created_at": "order_created_at", "markethashname": "marketHashName"})
    combined_df["priceUpdatedAt"] = pd.to_datetime(pd.Timestamp.now("UTC"), format='ISO8601', utc=True)  

    return combined_df

In [141]:
web_api_all_items_file_0 = json.loads(open("data/steam_web_api/api_item_game=cs2&page=0.json").read())
web_items_0 = getJsonObjAsFlatMappedDataFrame(web_api_all_items_file_0) 
web_items_normalized = normalize_web_api_items(web_items_0)

In [146]:
alternative_price_stats = get_aggregated_price_info(web_items_normalized)
extreme_spreads = alternative_price_stats[alternative_price_stats['extreme_spreads']]
print(f"Items with spread between Platforms over over 100% {extreme_spreads.shape[0]}")
print(f"Items without Spread: {alternative_price_stats[alternative_price_stats['rel_spread_pct'] <= 0].shape[0]}")
print(f"Average Spread across all items: {alternative_price_stats['rel_spread_pct'].sum() / alternative_price_stats['rel_spread_pct'].shape[0]}")  
alternative_price_stats.describe()

Items with spread between Platforms over over 100% 21173
Items without Spread: 907
Average Spread across all items: 1290.9306411747461


,market_count,min_price,median_price,max_price,std_price,abs_spread,rel_spread_pct
count,35028.000000,35028.000000,35028.000000,35028.000000,34134.000000,35028.000000,35028.000000
mean,10.412099,88.221637,104.018529,160.605533,28.427266,72.383896,1290.930641
std,3.903474,828.248895,1136.926467,1335.882681,340.086860,707.735717,6726.944185
min,1.000000,0.010000,0.010000,0.020000,0.000000,0.000000,0.000000
25%,8.000000,0.390000,0.570000,2.000000,0.335419,0.950000,44.784243
50%,12.000000,2.720000,3.775000,10.965000,1.863835,5.240000,178.894872
75%,14.000000,24.340000,30.902500,58.070000,8.484844,24.950000,801.384283
max,16.000000,74985.000000,149970.010000,157226.680000,46909.708569,90477.580000,349900.000000


# Same problem, look at Claude, small cent_prices make the spread look huge

In [ ]:
# TODO: adapt the calculations so it makes sense

# How off are prices of current actual values
-> test manually

# 4. Aktualität

Wie alt sind die Angebote? 

In [12]:

time_analyze_df_alt = verkaufs_angebote_alt.copy()
time_analyze_df_neu = verkaufs_angebote_neu.copy()

time_analyze_df_alt["time_between_capture_and_request"] = dt_verkaufs_angebote_alt - time_analyze_df_alt['capturedAt'] #dt_verkaufs_angebote_alt =  Timestamp('2026-11-17 13:44:47+0000', tz='UTC')
time_analyze_df_neu["time_between_capture_and_request"] = dt_verkaufs_angebote_neu - time_analyze_df_neu['capturedAt'] #dt_verkaufs_angebote_neu =  Timestamp('2026-11-17 18:05:45+0000', tz='UTC')


In [13]:
print(time_analyze_df_alt['time_between_capture_and_request'].quantile(0.99))
time_analyze_df_alt['time_between_capture_and_request'].describe()

0 days 00:13:57.402000


count                       392998
mean     0 days 00:03:04.746198263
std      0 days 00:03:26.263127807
min         0 days 00:00:21.532000
25%         0 days 00:00:45.627000
50%         0 days 00:02:11.386000
75%         0 days 00:03:40.457000
max         0 days 01:31:06.026000
Name: time_between_capture_and_request, dtype: object

In [14]:
print(time_analyze_df_neu['time_between_capture_and_request'].quantile(0.99))
time_analyze_df_neu['time_between_capture_and_request'].describe()

0 days 00:05:34.614000


count                       392415
mean     0 days 00:02:46.699852729
std      0 days 00:01:57.957954484
min         0 days 00:00:23.994000
25%         0 days 00:01:17.320000
50%         0 days 00:02:37.376000
75%         0 days 00:04:03.199000
max         0 days 01:52:04.362000
Name: time_between_capture_and_request, dtype: object

# 5. Stabilität über die Zeit (Stability over Time)

Bleiben API responses consistent über unterschiedliche Aufrufe

In [ ]:
# -> durch obere Analysen bewiesen

# 6. Volatilität 

Preis bzw Demandänderungen über die zeit

In [15]:
combined = pd.concat([verkaufs_angebote_alt,verkaufs_angebote_neu], axis=0)
combined.index = range(len(combined))
print(f"Combined = {combined.shape[0]}, addition = {verkaufs_angebote_alt.shape[0] + verkaufs_angebote_neu.shape[0]}")

price_stats = (combined
            .groupby(["marketHashName", "variant", "market"], dropna=False)['price'].agg(first_price = 'first', last_price = 'last').reset_index())

price_stats = price_stats[price_stats['first_price'] != price_stats['last_price']]


quantity_stats = (combined
            .groupby(["marketHashName", "variant", "market"], dropna=False)['quantity'].agg(first_quantity = 'first', last_quantity = 'last').reset_index())

quantity_stats = quantity_stats[quantity_stats['first_quantity'] != quantity_stats['last_quantity']]

Combined = 785413, addition = 785413


In [16]:
count_price_increase = price_stats[price_stats['first_price'] < price_stats['last_price']].shape[0]
count_price_decrease = price_stats[price_stats['first_price'] > price_stats['last_price']].shape[0]
count_quantity_increase = quantity_stats[quantity_stats['first_quantity'] < quantity_stats['last_quantity']].shape[0]
count_quantity_decrease = quantity_stats[quantity_stats['first_quantity'] > quantity_stats['last_quantity']].shape[0]



print(f"Price changes: {count_price_increase + count_price_decrease}")
print(f"Price increase: {count_price_increase}")
print(f"Price decreased: {count_price_decrease}")
print(f"Quantity changes: {count_quantity_decrease + count_quantity_increase}")
print(f"Quantity increased: {count_quantity_increase}")
print(f"Quantity decreased: {count_quantity_decrease}")

Price changes: 137853
Price increase: 60638
Price decreased: 77215
Quantity changes: 161188
Quantity increased: 63084
Quantity decreased: 98104


# Datenexploration der Ankaufsangebote

In [177]:
from pathlib import Path
import re

def get__files_by_pattern(parent_dir: Path, pattern_match: str )-> list[Path]:
    return sorted([x for x in parent_dir.iterdir() if parent_dir.is_dir() and x.is_file() and re.search(pattern_match, x.name)], key=lambda x: len(x.name))

def get_df_of_all_json_files_containing_pattern_match(parent_dir: Path, pattern_match: str)-> pd.DataFrame:
    files = get__files_by_pattern(parent_dir, pattern_match)
    df = pd.DataFrame()

    for file in files:
        file_data = json.loads(open(file).read())
        new_df = getJsonObjAsFlatMappedDataFrame(file_data["data"])
        df = pd.concat([df, new_df], axis=0)

    return df


def extract_buy_orders(df: pd.DataFrame):
    """
    markt dictionaries in buyOrders spalte werden exploded & normalisiert -> eine zeile pro markt, da es unterschiedliche namensgebungen zwischen den varianten markets und den no_variant markets gibt, werden die namen normalisiert (updatedAt / capturedAt -> capturedAt)
    """
    df = df.explode("buyOrders").dropna(subset=["buyOrders"]).reset_index(drop=True)
    market_details = pd.json_normalize(df["buyOrders"])
    return pd.concat(
        [
            df.drop(columns=["buyOrders"]),
            market_details.drop(columns=["icon"]),
        ],
        axis=1,
    )

def normalize_buy_orders(df: pd.DataFrame):
    df = df[["marketHashName","buyOrders"]]
    df = extract_buy_orders(df)
    df = df.drop(columns="sourceUpdatedAt")
    df["priceUpdatedAt"] = pd.to_datetime(pd.Timestamp.now("UTC"), format='ISO8601', utc=True)  
    return df


In [ ]:
data_folder = Path('./data/steam_data_api/')
old_pattern_match = r"buy_orders_page_[0-9]{1,2}_2026-11-17 13"

old_df = get_df_of_all_json_files_containing_pattern_match(data_folder, old_pattern_match)

In [179]:
old_df.head()

,marketHashName,spread,marketCount,demand,buyOrders,best.market,best.price,ask.market,ask.price,ask,best
0,Sticker | Titan (Holo) | Katowice 2014,4575114.0,7,28466.0,"[{'market': 'youpin', 'icon': 'https://steamda...",youpin,2100053.0,youpin,6675167.0,NaN,NaN
1,Sticker | Reason Gaming (Holo) | Katowice 2014,5400135.0,6,4162.0,"[{'market': 'youpin', 'icon': 'https://steamda...",youpin,2100053.0,buff163,7500188.0,NaN,NaN
2,EMS Katowice 2014 Challengers,354003.0,7,3131.0,"[{'market': 'csfloat', 'icon': 'https://steamd...",csfloat,1600000.0,skinport,1954003.0,NaN,NaN
3,EMS Katowice 2014 Legends,529950.0,7,4206.0,"[{'market': 'csfloat', 'icon': 'https://steamd...",csfloat,1600000.0,skinport,2129950.0,NaN,NaN
4,★ Sport Gloves | Hedge Maze (Factory New),1100068.0,6,167.0,"[{'market': 'csfloat', 'icon': 'https://steamd...",csfloat,1600000.0,youpin,2700068.0,NaN,NaN


In [180]:
old_df_normalized = normalize_buy_orders(old_df)
old_df_normalized.head()

,marketHashName,market,price,demand,capturedAt,priceUpdatedAt
0,Sticker | Titan (Holo) | Katowice 2014,youpin,2100053,NaN,2026-11-17T13:44:25.468Z,2026-11-18 14:15:31.117640+00:00
1,Sticker | Titan (Holo) | Katowice 2014,csfloat,2000000,155.0,2026-11-17T13:50:20.280Z,2026-11-18 14:15:31.117640+00:00
2,Sticker | Titan (Holo) | Katowice 2014,dmarket,604375,NaN,2026-11-17T13:43:40.743Z,2026-11-18 14:15:31.117640+00:00
3,Sticker | Titan (Holo) | Katowice 2014,steam,221061,28251.0,2026-11-17T13:48:09.454Z,2026-11-18 14:15:31.117640+00:00
4,Sticker | Titan (Holo) | Katowice 2014,buff163,180005,NaN,2026-11-17T13:44:25.468Z,2026-11-18 14:15:31.117640+00:00
